# Validation — Sentinel-1 flood map vs reference flood and silt extents

Compares the `final_flood` raster from `01_sentinel1_layer.ipynb` with independent(ish) reference products for Cyclone Gabrielle, and reports confusion-matrix accuracy. Section 8 compares the final map with k-means, an Otsu threshold and (optionally) the Sentinel-2 U-Net from `02_`, and makes the BIC, precision–recall and feature-space figures.

**Reads all AOI, dates and thresholds from `layer_config.py`.** Sections 1–2 rebuild `final_flood` with the same **2-D GMM** steps as `01_sentinel1_layer.ipynb` sections 1–4 (the UI `processing/pipeline.py` uses the same GMM; `MATCH_ORBIT` in `layer_config.py` keeps event and baseline on one orbit track). If the flood logic in `01_` changes, update section 2 here to match.

| Reference | Date | Source / method | Used for | Licence |
|---|---|---|---|---|
| Dragonfly Data Science, Hawke's Bay flood | 2023-02-14 | Sentinel-1 SAR (urban + non-urban layers) | Flood accuracy (section 4) | CC BY-SA 3.0 IGO |
| Dragonfly Data Science, Cyclone Gabrielle impact (HB + Gisborne) | 2023-02-19 / 21 | Sentinel-2 before/after; `category` = silt / slip / flood | Silt overlap (section 6) | CC BY-SA 3.0 IGO |
| Hawke's Bay Regional Council flood map | 13–15 Feb | Photos, video, imagery, inspections, contours | Visual check only (no download) | — |

Sources: [Dragonfly data release](https://www.dragonfly.co.nz/news/2023-02-17-cyclone-gabrielle) · [HBRC flood map](https://www.hbrc.govt.nz/our-council/cyclone-gabrielle-response/cyclone-gabrielle-flooding)

**Caveats**
- The Dragonfly 14 Feb flood layer is likely derived from the same Sentinel-1 acquisition used here (14 Feb 07:07 UTC). Agreement measures consistency between two SAR methods, not accuracy against ground truth. Treat scores as an upper bound.
- Reference products are model outputs, not field surveys.
- Run with **Kernel → Restart Kernel and Run All** so every number comes from one run.
- Downloads go to `data/raw/validation/` (already gitignored). Figures and CSVs go to `outputs/validation/`.

## 0. Load config and initialise Earth Engine

In [12]:
import sys, importlib, io, zipfile
from pathlib import Path
from urllib.request import urlopen, urlretrieve

import ee
import geemap
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from PIL import Image

_root = Path.cwd()
_nb = _root / "notebooks" if (_root / "notebooks" / "layer_config.py").exists() else _root
sys.path.insert(0, str(_nb))
import layer_config as cfg
importlib.reload(cfg)

_repo = _root if (_root / "processing" / "ee_auth.py").exists() else _root.parent
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from processing.ee_auth import initialize_ee

initialize_ee()

aoi = cfg.aoi_geometry()
SCALE = cfg.WORKING_SCALE
AREA_SCALE = 20                      # same as 01_ section 5b

RAW_DIR = _repo / "data" / "raw" / "validation"
OUT_DIR = _repo / "outputs" / "validation"
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

FLOOD_REF_URL = "https://files.dragonfly.co.nz/data/hawkes-bay-flood/hawkes-bay-flood-2023-02-14.zip"
IMPACT_REF_URL = "https://files.dragonfly.co.nz/data/hawkes-bay-flood/hawkes-bay-flood-silt-slip-2023-02-21.zip"

print("AOI:", cfg.MAP_CENTER, f"±{cfg.AOI_RADIUS_KM} km |", cfg.START_DATE, "→", cfg.END_DATE)

AOI: [-39.58, 176.88] ±5.0 km | 2023-02-12 → 2023-02-17


## 1. Sentinel-1 event and historical stacks
Same as `01_` sections 1, 1b, 2.

In [13]:
def s1_iw_vv_vh_angle(start, end):
    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.eq("resolution_meters", 10))
        .select(["VV", "VH", "angle"])
    )

# Ancillary layers
_fabdem = ee.ImageCollection("projects/sat-io/open-datasets/FABDEM").filterBounds(aoi)
dem = _fabdem.mosaic().select("b1").setDefaultProjection(_fabdem.first().projection()).clip(aoi).rename("elevation")
slope_deg = ee.Terrain.slope(dem).rename("slope")
worldcover = ee.Image("ESA/WorldCover/v200/2021").clip(aoi)
wc_water = worldcover.eq(80)
dark_land = ee.Image(0)
for _cls in cfg.EXCLUDE_WORLDCOVER_CLASSES:
    dark_land = dark_land.Or(worldcover.eq(_cls))
gsw = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").clip(aoi)
gsw_permanent = gsw.select("occurrence").gte(cfg.PERMANENT_WATER_OCCURRENCE).unmask(0)
ocean_no_dem = dem.mask().Not().unmask(1)
permanent_water = gsw_permanent.Or(wc_water).Or(ocean_no_dem).rename("permanent_water")

# Per scene: dB -> linear, Lee filter (3x3, ENL 4.9), angular terrain flattening
# (Vollrath et al. 2020, volume model, geometric layover/shadow mask); then the
# per-pixel median, back to dB. Shared with the UI: processing/s1_preprocess.py.
from processing.s1_preprocess import composite_db, lee_filter, terrain_geometry, to_db, to_natural

tf_slope, tf_phi_s = terrain_geometry(aoi)


def build_composite(col):
    return composite_db(col, aoi, tf_slope, tf_phi_s, enl=cfg.SENTINEL1_ENL)

co_col = s1_iw_vv_vh_angle(cfg.START_DATE, cfg.END_DATE)
from processing.s1_tracks import choose_track, filter_track

# Pick ONE orbit track: it must cover >=95% of the AOI, then it is the one whose
# scene is closest to FLOOD_PEAK_DATE. Event and historical stacks both use it, so
# ΔWI compares one viewing geometry. Falls back to mixing tracks (with a warning)
# if no single track covers the AOI, or if MATCH_ORBIT = False in layer_config.py.
track = choose_track(co_col, aoi, cfg.FLOOD_PEAK_DATE, match=cfg.MATCH_ORBIT)
print(track["message"])
closest = ee.Image(co_col.filter(ee.Filter.eq("system:index", track["closest_id"])).first()).set(
    "peak_diff_days", track["peak_diff_days"])
print("Scene closest to peak on that track:", track["closest_id"], f"| |Δdays| = {track['peak_diff_days']:.2f}")


def same_track(col):
    return filter_track(col, track)


co_col = same_track(co_col)
if track["matched"]:
    # Event image = the single pass nearest the peak (all its slices), not an average
    # of every pass in the window, which would dilute the flood with drained acquisitions.
    _t_peak = closest.date()
    co_col = co_col.filterDate(_t_peak.advance(-1, "hour"), _t_peak.advance(1, "hour"))
n_co = co_col.size().getInfo()
print("Event scenes used (slices of the peak pass):", n_co)
co_db = build_composite(co_col)

from datetime import date, timedelta
hist_start = (date.fromisoformat(cfg.START_DATE) - timedelta(days=int(round(365.25 * cfg.HIST_LOOKBACK_YEARS)))).isoformat()
hist_col = same_track(s1_iw_vv_vh_angle(hist_start, cfg.START_DATE))
print("Historical S1 scenes:", hist_col.size().getInfo(), "|", hist_start, "→", cfg.START_DATE)
hist_db = build_composite(hist_col)

Event S1 scenes: 2
Historical S1 scenes: 108 | 2022-02-12 → 2023-02-12


## 2. Rebuild `final_flood`
Same as `01_` sections 3, 3b, 4: 2-D GMM on standardised (`WI`, `ΔWI`), then WorldCover / JRC / DEM / MMU post-masks. Not Otsu or `WI_CHANGE_MIN_DB`.

In [14]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

N_CLUSTERS = 4
CLUSTER_SAMPLE_PIXELS = 8000
CLUSTER_SEED = 0
_GETINFO_CHUNK = 4000

water_index = co_db.select("VV").add(co_db.select("VH")).rename("WI")
hist_wi = hist_db.select("VV").add(hist_db.select("VH")).rename("WI_hist")
wi_change = hist_wi.subtract(water_index).rename("WI_change")

feat = water_index.addBands(wi_change).updateMask(permanent_water.Not())
feat_stats = feat.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
    geometry=aoi, scale=SCALE, bestEffort=True, maxPixels=1e9, tileScale=4,
).getInfo()
wi_mean, wi_std = feat_stats["WI_mean"], feat_stats["WI_stdDev"]
dw_mean, dw_std = feat_stats["WI_change_mean"], feat_stats["WI_change_stdDev"]
if wi_std is None or dw_std is None or wi_std == 0 or dw_std == 0:
    raise ValueError("Could not standardise WI / ΔWI — check AOI coverage.")

wi_z = water_index.subtract(wi_mean).divide(wi_std).rename("WI_z")
dw_z = wi_change.subtract(dw_mean).divide(dw_std).rename("dWI_z")
feat_z = wi_z.addBands(dw_z).updateMask(permanent_water.Not()).clip(aoi)
training = feat_z.sample(
    region=aoi, scale=SCALE, numPixels=CLUSTER_SAMPLE_PIXELS,
    seed=CLUSTER_SEED, geometries=False, tileScale=4,
)

def _fc_props_to_array(fc, props, chunk=_GETINFO_CHUNK):
    n = int(fc.size().getInfo() or 0)
    rows = []
    for start in range(0, n, chunk):
        feats = ee.FeatureCollection(fc.toList(chunk, start)).getInfo()["features"]
        for f in feats:
            rows.append([f["properties"].get(p) for p in props])
    return np.array(rows, dtype=float)

X = _fc_props_to_array(training, ["WI_z", "dWI_z"])
X = X[np.isfinite(X).all(axis=1)]
print(f"Cluster training samples: {len(X)} (permanent water excluded)")
assert len(X) >= N_CLUSTERS * 20, "Too few samples for GMM / k-means."

gmm = GaussianMixture(
    n_components=N_CLUSTERS, covariance_type="full",
    random_state=CLUSTER_SEED, n_init=5,
).fit(X)
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=CLUSTER_SEED, n_init=10).fit(X)

def _component_table(means_z, weights=None):
    wi = means_z[:, 0] * wi_std + wi_mean
    dw = means_z[:, 1] * dw_std + dw_mean
    rows = {"WI_mean_dB": wi, "dWI_mean_dB": dw, "score": dw / dw_std - means_z[:, 0]}
    if weights is not None:
        rows["weight"] = weights
    return pd.DataFrame(rows)

def _pick_flood_id(table):
    changed = table[table["dWI_mean_dB"] > 0]
    pool = changed if len(changed) else table
    return int(pool["score"].idxmax())

gmm_table = _component_table(gmm.means_, gmm.weights_)
kmeans_table = _component_table(kmeans.cluster_centers_)
gmm_flood_id = _pick_flood_id(gmm_table)
kmeans_flood_id = _pick_flood_id(kmeans_table)
print("GMM components\n", gmm_table.round(3))
print("GMM flood component:", gmm_flood_id)
print("K-means centroids\n", kmeans_table.round(3))
print("K-means flood cluster:", kmeans_flood_id)

def _gmm_loglik_image(k):
    mean, cov, weight = gmm.means_[k], gmm.covariances_[k], float(gmm.weights_[k])
    prec = np.linalg.inv(cov)
    det = float(np.linalg.det(cov))
    log_norm = np.log(max(weight, 1e-12)) - 0.5 * np.log((2.0 * np.pi) ** 2 * max(det, 1e-12))
    d0 = wi_z.subtract(float(mean[0]))
    d1 = dw_z.subtract(float(mean[1]))
    maha = (
        d0.pow(2).multiply(float(prec[0, 0]))
        .add(d0.multiply(d1).multiply(2.0 * float(prec[0, 1])))
        .add(d1.pow(2).multiply(float(prec[1, 1])))
    )
    return ee.Image.constant(log_norm).subtract(maha.multiply(0.5)).rename(f"ll_{k}")

def _kmeans_negdist_image(k):
    mean = kmeans.cluster_centers_[k]
    d0 = wi_z.subtract(float(mean[0]))
    d1 = dw_z.subtract(float(mean[1]))
    return d0.pow(2).add(d1.pow(2)).multiply(-1).rename(f"nd_{k}")

def _argmax_label(score_images):
    best = score_images[0]
    label = ee.Image.constant(0)
    for k, img in enumerate(score_images[1:], start=1):
        label = label.where(img.gt(best), k)
        best = best.max(img)
    return label.rename("cluster")

gmm_flood = _argmax_label([_gmm_loglik_image(k) for k in range(N_CLUSTERS)]).eq(gmm_flood_id).rename("gmm_flood")
kmeans_flood = _argmax_label([_kmeans_negdist_image(k) for k in range(N_CLUSTERS)]).eq(kmeans_flood_id).rename("kmeans_flood")

# WorldCover / JRC / DEM are post-masks, not GMM features (same as 01_ 3b / 4).
flood_candidate = gmm_flood.And(permanent_water.Not()).And(dark_land.Not()).rename("flood_candidate")

dem_pct = dem.reduceRegion(ee.Reducer.percentile([5, 50, 95]), aoi, 30, bestEffort=True, maxPixels=1e9).getInfo()
elev_ok = dem.lt(cfg.ELEVATION_MAX_M).unmask(1)
if cfg.ELEVATION_ABOVE_P5_M is not None:
    elev_ok = elev_ok.And(dem.lte(dem_pct["elevation_p5"] + cfg.ELEVATION_ABOVE_P5_M).unmask(1))
slope_ok = slope_deg.lt(cfg.SLOPE_MAX_DEG).unmask(1)
# Active layover/shadow from the terrain-flattening geometry: masked in the composite.
layover_shadow_risk = co_db.select("VV").mask().Not().rename("layover_shadow_risk")

flood_masked = flood_candidate.And(elev_ok).And(slope_ok).And(layover_shadow_risk.Not())
blob_size = flood_masked.selfMask().connectedPixelCount(maxSize=128, eightConnected=True)
flood_mmu = flood_masked.updateMask(blob_size.gte(cfg.MMU_PIXELS)).unmask(0)
final_flood = (flood_mmu.focal_min(radius=SCALE, units="meters")
               .focal_max(radius=SCALE, units="meters").clip(aoi).rename("flood"))

px_km2 = ee.Image.pixelArea().divide(1e6)
valid = permanent_water.Not()

def area_km2(mask_img, only_valid=True):
    m = mask_img.And(valid) if only_valid else mask_img
    v = px_km2.updateMask(m).reduceRegion(ee.Reducer.sum(), aoi, AREA_SCALE,
                                          bestEffort=True, maxPixels=1e10).get("area").getInfo()
    return 0.0 if v is None else v

print("gmm_flood km²: %.3f | kmeans_flood km²: %.3f" % (
    area_km2(gmm_flood, only_valid=False), area_km2(kmeans_flood, only_valid=False)))
print("final_flood (km², should match 01_ flood_final_km2): %.3f" % area_km2(final_flood, only_valid=False))


Cluster training samples: 7363 (permanent water excluded)
GMM components
    WI_mean_dB  dWI_mean_dB  score  weight
0     -36.810        7.039  1.371   0.133
1     -50.109       19.920  3.954   0.156
2     -23.457       -2.894 -0.905   0.173
3     -24.607       -2.941 -0.804   0.538
GMM flood component: 1
K-means centroids
    WI_mean_dB  dWI_mean_dB  score
0     -20.343       -6.177 -1.538
1     -50.477       20.478  4.047
2     -26.846       -0.940 -0.387
3     -38.643        8.472  1.691
K-means flood cluster: 1
gmm_flood km²: 14.680 | kmeans_flood km²: 14.326
final_flood (km², should match 01_ flood_final_km2): 14.135


## 3. Reference flood extent (Dragonfly, 14 Feb)
The zip holds two layers, urban and non-urban flood. Both are kept and tagged.

In [15]:
def fetch_zip(url, name):
    zpath = RAW_DIR / f"{name}.zip"
    if not zpath.exists():
        urlretrieve(url, zpath)
    out = RAW_DIR / name
    if not out.exists():
        zipfile.ZipFile(zpath).extractall(out)
    return out

flood_dir = fetch_zip(FLOOD_REF_URL, "dragonfly_flood_20230214")
parts = []
for name in ["non_urban_flood_nztm.gpkg", "urban_flood_nztm.gpkg"]:
    g = gpd.read_file(next(flood_dir.rglob(name))).to_crs(4326)
    g["part"] = name.replace("_nztm.gpkg", "")
    parts.append(g)
ref = gpd.clip(pd.concat(parts, ignore_index=True), cfg.AOI_BOUNDS)
ref = ref[~ref.geometry.is_empty].copy()
ref["geometry"] = ref.geometry.simplify(0.00005)
print(ref["part"].value_counts())

def to_mask(gdf):
    return ee.Image(0).paint(geemap.gdf_to_ee(gdf[["geometry"]]), 1).clip(aoi)

ref_img = to_mask(ref)

part
non_urban_flood    157
urban_flood         25
Name: count, dtype: int64


## 4. Confusion matrix and scores
Permanent water is excluded from both maps, so scores cover new inundation only.

In [16]:
code_img = final_flood.multiply(2).add(ref_img).updateMask(valid).rename("code")  # 0 TN, 1 FN, 2 FP, 3 TP
groups = px_km2.addBands(code_img).reduceRegion(
    ee.Reducer.sum().group(groupField=1, groupName="code"),
    aoi, AREA_SCALE, bestEffort=True, maxPixels=1e10,
).get("groups").getInfo()
km2 = {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0}
for g in groups:
    km2[int(g["code"])] = g["sum"]
TN, FN, FP, TP = km2[0], km2[1], km2[2], km2[3]

precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)
iou = TP / (TP + FP + FN)
N_ALL = TP + TN + FP + FN
p_obs = (TP + TN) / N_ALL
# Chance agreement from the row/column totals (area-weighted, same as pixel counts at fixed scale).
p_exp = ((TP + FP) * (TP + FN) + (FN + TN) * (FP + TN)) / N_ALL ** 2
kappa = (p_obs - p_exp) / (1 - p_exp)

cm = pd.DataFrame([[TP, FP], [FN, TN]], index=["S1: flood", "S1: dry"],
                  columns=["Reference: flood", "Reference: dry"]).round(3)
scores = pd.Series({
    "S1 flood (km²)": TP + FP, "Reference flood (km²)": TP + FN,
    "Precision": precision, "Recall": recall, "F1": f1, "IoU": iou,
    "Overall accuracy": p_obs, "Cohen's kappa": kappa,
}).round(3)

rows = []
for part in ["non_urban_flood", "urban_flood"]:
    sub = ref[ref["part"] == part]
    if len(sub) == 0:
        continue
    m = to_mask(sub)
    r_km2, hit = area_km2(m), area_km2(m.And(final_flood))
    rows.append({"reference type": part, "reference km²": r_km2, "found km²": hit,
                 "recall": hit / r_km2 if r_km2 else np.nan})
by_type = pd.DataFrame(rows).round(3)

print("Confusion matrix (km²)\n", cm, "\n")
print(scores, "\n")
print(by_type)

Confusion matrix (km²)
            Reference: flood  Reference: dry
S1: flood            13.562           0.574
S1: dry               7.887          69.832 

S1 flood (km²)           14.135
Reference flood (km²)    21.448
Precision                 0.959
Recall                    0.632
F1                        0.762
IoU                       0.616
Overall accuracy          0.908
dtype: float64 

    reference type  reference km²  found km²  recall
0  non_urban_flood         20.859     13.562   0.650
1      urban_flood          0.610      0.001   0.002


## 5. Agreement map

In [17]:
agree = code_img.remap([1, 2, 3], [1, 2, 3]).selfMask()
AGREE_VIS = {"min": 1, "max": 3, "palette": ["#2c7bb6", "#d7191c", "#1a9641"]}

Map = geemap.Map(center=cfg.MAP_CENTER, zoom=12, basemap="HYBRID")
Map.addLayer(agree, AGREE_VIS, "Agreement (blue = reference only, red = S1 only, green = both)")
Map

Map(center=[-39.58, 176.88], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

In [ ]:
%matplotlib inline
from matplotlib.ticker import FuncFormatter

W, S, E, N = cfg.AOI_BOUNDS
lat0 = (S + N) / 2
km_per_deg_lon = 111.32 * np.cos(np.radians(lat0))

# Background = event Sentinel-1 VV (grey); permanent water pale blue; agreement classes on top.
bg = co_db.select("VV").focal_median(1, "square", "pixels").visualize(min=-22, max=0)  # 3x3 median hides the frame seam (display only)
pw = permanent_water.selfMask().visualize(palette=["#c6dbef"])
rgb = bg.blend(pw).blend(agree.visualize(**AGREE_VIS))
from urllib.error import HTTPError


def ee_png(img, region, sizes=(1200, 900, 600)):
    """Fetch an EE thumbnail, stepping down in size if EE refuses (time/memory limits).

    ~1000 px across a 10 km AOI is about the 10 m analysis resolution; larger
    sizes recompute the whole chain finer than the analysis and can time out.
    """
    for dims in sizes:
        url = img.getThumbURL({"region": region, "dimensions": dims, "format": "png", "crs": "EPSG:4326"})
        try:
            return np.array(Image.open(io.BytesIO(urlopen(url).read())))
        except HTTPError as err:
            print(f"Thumbnail at {dims}px refused ({err.code}): {err.read().decode(errors='ignore')[:300]}")
    raise RuntimeError("Earth Engine refused every thumbnail size; see the messages above.")


arr = ee_png(rgb, aoi)

fig, ax = plt.subplots(figsize=(7.5, 7.5))
ax.imshow(arr, extent=[W, E, S, N], interpolation="nearest")
ax.set_aspect(1 / np.cos(np.radians(lat0)))   # equal ground distance in x and y
ax.set_xlim(W, E); ax.set_ylim(S, N)
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.2f}°E"))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{abs(v):.2f}°S"))
ax.tick_params(labelsize=8)
for lab in ax.get_yticklabels():
    lab.set_rotation(90); lab.set_va("center")

# ESRI-style alternating scale bar + north arrow (shared with the app figure).
from processing.map_furniture import add_north_arrow, add_scale_bar
add_scale_bar(ax, km_per_deg_lon)
add_north_arrow(ax, loc="upper left")

ax.legend(handles=[Patch(color="#1a9641", label="Both flood (TP)"),
                   Patch(color="#d7191c", label="Sentinel-1 only (FP)"),
                   Patch(color="#2c7bb6", label="Reference only (FN)"),
                   Patch(facecolor="#c6dbef", edgecolor="0.4", label="Permanent water (excluded)")],
          loc="lower right", fontsize=8, framealpha=0.92)
fig.savefig(OUT_DIR / "agreement.png", dpi=300, bbox_inches="tight")
plt.show()

# Paste this as the caption in the report (not drawn on the figure).
print(
    f"Agreement between the Sentinel-1 GMM flood map ({cfg.START_DATE} to {cfg.END_DATE}) and the "
    f"Dragonfly Data Science 14 February 2023 flood extent (CC BY-SA 3.0 IGO), Cyclone Gabrielle, Hawke's Bay. "
    f"AOI ±{cfg.AOI_RADIUS_KM:g} km around {cfg.MAP_CENTER[0]:.2f}°, {cfg.MAP_CENTER[1]:.2f}°. "
    f"Background: event Sentinel-1 VV backscatter, 3×3 median for display (Copernicus Sentinel data 2023). WGS 84 (EPSG:4326). "
    f"IoU = {iou:.2f}, F1 = {f1:.2f}, kappa = {kappa:.2f}. Permanent water excluded from scoring."
)

## 6. Silt overlap (Dragonfly impact layer, 19/21 Feb)
Tests whether S1 false alarms sit on silt-covered ground. Silt and flood overlap spatially, so a high share is consistent with wet silt being mapped as water but does not prove it. Baseline for comparison: silt share of the AOI.

In [10]:
impact_dir = fetch_zip(IMPACT_REF_URL, "dragonfly_impact_20230221")
impact = gpd.read_file(next(impact_dir.rglob("*.gpkg"))).to_crs(4326)
impact = gpd.clip(impact, cfg.AOI_BOUNDS)
print(impact["category"].value_counts(), "\n")

silt = impact[impact["category"] == "silt"].copy()
silt["geometry"] = silt.geometry.simplify(0.00005)
silt_img = to_mask(silt)

fp_img = final_flood.And(ref_img.Not())
aoi_valid_km2 = area_km2(ee.Image(1).clip(aoi))
silt_stats = pd.Series({
    "Silt area (km²)": area_km2(silt_img),
    "Silt share of AOI": area_km2(silt_img) / aoi_valid_km2,
    "S1 flood on silt / S1 flood": area_km2(final_flood.And(silt_img)) / area_km2(final_flood),
    "Reference flood on silt / reference flood": area_km2(ref_img.And(silt_img)) / area_km2(ref_img),
    "S1 false alarm on silt / S1 false alarm": area_km2(fp_img.And(silt_img)) / area_km2(fp_img),
}).round(3)
print(silt_stats)

Map2 = geemap.Map(center=cfg.MAP_CENTER, zoom=12, basemap="HYBRID")
Map2.addLayer(silt_img.selfMask(), {"palette": ["#a6611a"]}, "Dragonfly silt 19/21 Feb")
Map2.addLayer(agree, AGREE_VIS, "Agreement", False)
Map2

category
silt     898
flood    102
slip      31
Name: count, dtype: int64 

Silt area (km²)                              30.941
Silt share of AOI                             0.337
S1 flood on silt / S1 flood                   0.813
Reference flood on silt / reference flood     0.790
S1 false alarm on silt / S1 false alarm       0.731
dtype: float64


Map(center=[-39.58, 176.88], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

## 7. Save tables

In [11]:
cm.to_csv(OUT_DIR / "confusion_km2.csv")
scores.to_csv(OUT_DIR / "scores.csv")
by_type.to_csv(OUT_DIR / "recall_by_type.csv", index=False)
silt_stats.to_csv(OUT_DIR / "silt_overlap.csv")
print("Saved to", OUT_DIR)

Saved to /Users/dongwook.kim/UoA/GEOG761-GIS/outputs/validation


## 8. Method comparison

Scores the final map against alternatives on the **same reference, permanent-water exclusion and 20 m area statistics as section 4**, so every row is directly comparable.

| Row | What it is |
|---|---|
| GMM + masks (final) | `final_flood` from section 2 |
| GMM / k-means, cluster only | The raw flood cluster, no post-masks |
| k-means + masks | k-means flood cluster with the same post-masks as the final map (elevation, slope, layover/shadow, MMU, opening) |
| Otsu on WI | One automatic threshold on the event water index (Otsu, 1979), no learning. Histogram over the same pixels as the GMM fit (permanent water excluded). Shown raw and with the same post-masks |
| Earlier rule + masks | The project's pre-GMM rule, `WI < min(Otsu, WATER_INDEX_MAX_DB)` and `ΔWI > WI_CHANGE_MIN_DB`, applied to the current preprocessing with the same post-masks |
| Sentinel-2 U-Net (optional) | The flood GeoTIFF from `02_sentinel2_layer.ipynb`, scored on its own 10 m grid. **Run notebook 02 first**; skipped otherwise. "Clear sky" scores only pixels with Cloud Score+ `cs_cdf ≥ 0.6` |

Also computed: GMM BIC and k-means inertia for 2–8 clusters, a precision–recall curve as the GMM flood-probability cut-off varies (each cut-off gets the same post-masks), and the feature-space plot of the clustering sample.

Outputs in `outputs/validation/`: `method_comparison.csv`, `pr_curve.csv`, `cluster_choice.csv`, `method_comparison.png`, `feature_space.png`. Takes a few minutes: each row and each cut-off is one Earth Engine reduction.

### 8a. Sentinel-1 variants (GMM, k-means, Otsu)

In [ ]:
from processing.map_furniture import FLOOD_COLOR


def metrics_from_counts(tp, fp, fn, tn):
    """Confusion areas -> the same scores as section 4."""
    n = tp + fp + fn + tn
    p_obs = (tp + tn) / n
    p_exp = ((tp + fp) * (tp + fn) + (fn + tn) * (fp + tn)) / n ** 2
    return {
        "Flood mapped (km²)": tp + fp,
        "Precision": tp / (tp + fp) if tp + fp else np.nan,
        "Recall": tp / (tp + fn) if tp + fn else np.nan,
        "F1": 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else np.nan,
        "IoU": tp / (tp + fp + fn) if tp + fp + fn else np.nan,
        "Kappa": (p_obs - p_exp) / (1 - p_exp) if p_exp < 1 else np.nan,
        "Overall accuracy": p_obs,
    }


urban_ref = ref[ref["part"] == "urban_flood"]
urban_img = to_mask(urban_ref) if len(urban_ref) else None
urban_km2 = area_km2(urban_img) if urban_img is not None else 0.0


def score_flood(flood_img, urban=True):
    """Area-based confusion matrix against ref_img, permanent water excluded (as section 4)."""
    flood_img = flood_img.unmask(0).gt(0)
    code = flood_img.multiply(2).add(ref_img).updateMask(valid).rename("code")  # 0 TN, 1 FN, 2 FP, 3 TP
    groups = px_km2.addBands(code).reduceRegion(
        ee.Reducer.sum().group(groupField=1, groupName="code"),
        aoi, AREA_SCALE, bestEffort=True, maxPixels=1e10,
    ).get("groups").getInfo()
    k = {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0}
    for g in groups:
        k[int(g["code"])] = g["sum"]
    out = metrics_from_counts(tp=k[3], fp=k[2], fn=k[1], tn=k[0])
    if urban and urban_km2:
        out["Urban recall"] = area_km2(urban_img.And(flood_img)) / urban_km2
    return out


def apply_post_masks(cluster_flood):
    """The section 2 post-masks, for any raw 0/1 flood image."""
    cand = cluster_flood.And(permanent_water.Not()).And(dark_land.Not())
    masked = cand.And(elev_ok).And(slope_ok).And(layover_shadow_risk.Not())
    blob = masked.selfMask().connectedPixelCount(maxSize=128, eightConnected=True)
    mmu = masked.updateMask(blob.gte(cfg.MMU_PIXELS)).unmask(0)
    return (mmu.focal_min(radius=SCALE, units="meters")
            .focal_max(radius=SCALE, units="meters").clip(aoi).rename("flood"))


def otsu_threshold(counts, bucket_min, bucket_width):
    """Otsu (1979): the split that maximises between-class variance. Returns the bucket edge."""
    c = np.asarray(counts, dtype=float)
    centres = bucket_min + bucket_width * (np.arange(len(c)) + 0.5)
    w0 = np.cumsum(c)[:-1]
    w1 = c.sum() - w0
    s0 = np.cumsum(c * centres)[:-1]
    with np.errstate(invalid="ignore", divide="ignore"):
        m0 = s0 / w0
        m1 = ((c * centres).sum() - s0) / w1
        between = w0 * w1 * (m0 - m1) ** 2
    i = int(np.nanargmax(between))
    return float(bucket_min + bucket_width * (i + 1))


# Sanity check: the helper reproduces the final map exactly.
assert abs(area_km2(apply_post_masks(gmm_flood), only_valid=False)
           - area_km2(final_flood, only_valid=False)) < 1e-3, "apply_post_masks differs from section 2"

# Otsu threshold on the event WI, same pixels as the GMM fit (permanent water excluded).
wi_hist = water_index.updateMask(valid).reduceRegion(
    ee.Reducer.histogram(maxBuckets=256, minBucketWidth=0.1),
    aoi, SCALE, bestEffort=True, maxPixels=1e9, tileScale=4,
).get("WI").getInfo()
otsu_wi = otsu_threshold(wi_hist["histogram"], wi_hist["bucketMin"], wi_hist["bucketWidth"])
earlier_cut = min(otsu_wi, cfg.WATER_INDEX_MAX_DB)
print(f"Otsu(WI) = {otsu_wi:.2f} dB | earlier rule: WI < {earlier_cut:.2f} dB and ΔWI > {cfg.WI_CHANGE_MIN_DB} dB")

otsu_raw = water_index.lt(otsu_wi).rename("otsu")
earlier_raw = water_index.lt(earlier_cut).And(wi_change.gt(cfg.WI_CHANGE_MIN_DB)).rename("earlier")

variants = {
    "S1 GMM + masks (final)": final_flood,
    "S1 GMM, cluster only": gmm_flood,
    "S1 k-means + masks": apply_post_masks(kmeans_flood),
    "S1 k-means, cluster only": kmeans_flood,
    "S1 Otsu on WI + masks": apply_post_masks(otsu_raw),
    "S1 Otsu on WI, threshold only": otsu_raw,
    "S1 earlier rule (Otsu + cap + ΔWI) + masks": apply_post_masks(earlier_raw),
}
rows = {}
for name, img in variants.items():
    rows[name] = score_flood(img)
    print(f"  scored: {name}")
comparison = pd.DataFrame(rows).T

# The final row must match section 4.
assert abs(comparison.loc["S1 GMM + masks (final)", "IoU"] - iou) < 1e-6, "final row does not match section 4"

### 8b. Sentinel-2 U-Net (optional, needs notebook 02 output)

In [ ]:
# Sentinel-2 U-Net rows (optional): needs notebooks/_s2_unet_cache/ from notebook 02.
import rasterio
from rasterio.enums import Resampling
from rasterio.features import rasterize
from datetime import date, timedelta

S2_CACHE = _repo / "notebooks" / "_s2_unet_cache"
s2_flood_tif = S2_CACHE / "s2_unet_flood.tif"
s2_event_tif = S2_CACHE / "event_l1c.tif"
CLEAR_CS_MIN = 0.6  # Cloud Score+ cs_cdf; >= 0.6 counts as clear sky


def ee_on_grid(img, band, transform, crs, shape):
    """Sample an Earth Engine image onto a local raster grid (same pixels as the GeoTIFF)."""
    arr = ee.data.computePixels({
        "expression": img.select([band]).toFloat(),
        "fileFormat": "NUMPY_NDARRAY",
        "grid": {
            "dimensions": {"width": int(shape[1]), "height": int(shape[0])},
            "affineTransform": {
                "scaleX": transform.a, "shearX": transform.b, "translateX": transform.c,
                "shearY": transform.d, "scaleY": transform.e, "translateY": transform.f,
            },
            "crsCode": crs.to_string(),
        },
    })
    return np.asarray(arr[band], dtype=float)


def burn(gdf, crs, transform, shape):
    if len(gdf) == 0:
        return np.zeros(shape, dtype=bool)
    shapes = ((geom, 1) for geom in gdf.to_crs(crs).geometry if geom is not None and not geom.is_empty)
    return rasterize(shapes, out_shape=shape, transform=transform, fill=0, dtype="uint8") > 0


def score_on_grid(pred, ref_g, urban_g, keep, px_area_km2):
    p, r = pred & keep, ref_g & keep
    tp, fp, fn = (p & r).sum(), (p & ~r).sum(), (~p & r).sum()
    tn = (keep & ~p & ~r).sum()
    out = metrics_from_counts(*(float(v) * px_area_km2 for v in (tp, fp, fn, tn)))
    u = urban_g & keep
    out["Urban recall"] = (p & u).sum() / u.sum() if u.sum() else np.nan
    return out


s2_rows = {}
if not s2_flood_tif.exists():
    print(f"No {s2_flood_tif.name}: run 02_sentinel2_layer.ipynb first. Sentinel-2 rows skipped.")
else:
    with rasterio.open(s2_flood_tif) as src:
        s2_flood = src.read(1) > 0
        g_t, g_crs, g_shape = src.transform, src.crs, src.shape
    px_area = abs(g_t.a * g_t.e) / 1e6
    with rasterio.open(s2_event_tif) as src:
        bands = src.read(out_shape=(src.count, *g_shape), resampling=Resampling.nearest).astype(float)
    s2_has_data = np.all(np.isfinite(bands), axis=0) & np.any(bands != 0, axis=0)

    # Same scene notebook 02 uses: first L1C on or after the flood peak.
    _s2_end = (date.fromisoformat(cfg.END_DATE) + timedelta(days=20)).isoformat()
    s2_scene = ee.Image(
        ee.ImageCollection("COPERNICUS/S2_HARMONIZED").filterBounds(aoi)
        .filterDate(cfg.FLOOD_PEAK_DATE, _s2_end).sort("system:time_start").first()
    )
    s2_id = s2_scene.get("system:index").getInfo()
    s2_when = ee.Date(s2_scene.get("system:time_start")).format("YYYY-MM-dd HH:mm").getInfo()
    print(f"Sentinel-2 scene: {s2_id} ({s2_when} UTC)")

    perm_g = ee_on_grid(permanent_water, "permanent_water", g_t, g_crs, g_shape) > 0.5
    ref_g = burn(ref, g_crs, g_t, g_shape)
    urban_g = burn(urban_ref, g_crs, g_t, g_shape)
    keep_all = s2_has_data & ~perm_g

    s2_rows["S2 U-Net, whole scene"] = score_on_grid(s2_flood, ref_g, urban_g, keep_all, px_area)

    cs_col = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED").filter(
        ee.Filter.eq("system:index", s2_id))
    if cs_col.size().getInfo():
        cs = ee_on_grid(cs_col.first(), "cs_cdf", g_t, g_crs, g_shape)
        keep_clear = keep_all & (cs >= CLEAR_CS_MIN)
        print(f"Clear sky (cs_cdf ≥ {CLEAR_CS_MIN}): {keep_clear.sum() / max(keep_all.sum(), 1):.0%} of scored pixels")
        s2_rows["S2 U-Net, clear sky only"] = score_on_grid(s2_flood, ref_g, urban_g, keep_clear, px_area)
    else:
        print("No Cloud Score+ image for this scene; clear-sky row skipped.")

### 8c. Comparison table

In [ ]:
method_table = comparison.copy()
if s2_rows:
    method_table = pd.concat([method_table, pd.DataFrame(s2_rows).T])
method_table = method_table[
    ["Flood mapped (km²)", "Precision", "Recall", "F1", "IoU", "Kappa", "Urban recall", "Overall accuracy"]
].astype(float).round(3)
method_table.to_csv(OUT_DIR / "method_comparison.csv")
method_table

### 8d. Number of clusters and precision–recall

In [ ]:
# (a) How many clusters? GMM BIC and k-means inertia on the same sample X.
K_RANGE = range(2, 9)
cluster_choice = pd.DataFrame({
    "k": list(K_RANGE),
    "GMM BIC": [GaussianMixture(n_components=k, covariance_type="full", random_state=CLUSTER_SEED,
                                n_init=5).fit(X).bic(X) for k in K_RANGE],
    "k-means inertia": [KMeans(n_clusters=k, random_state=CLUSTER_SEED, n_init=10).fit(X).inertia_
                        for k in K_RANGE],
})
cluster_choice.to_csv(OUT_DIR / "cluster_choice.csv", index=False)
print(cluster_choice.round(1).to_string(index=False))
print("Lowest BIC at k =", int(cluster_choice.loc[cluster_choice["GMM BIC"].idxmin(), "k"]),
      f"| used k = {N_CLUSTERS}")

# (b) Precision-recall: GMM posterior P(flood component | pixel), thresholded, then the same post-masks.
ll = [_gmm_loglik_image(k) for k in range(N_CLUSTERS)]
ll_max = ll[0]
for img in ll[1:]:
    ll_max = ll_max.max(img)
denom = ee.Image.constant(0)
for img in ll:
    denom = denom.add(img.subtract(ll_max).exp())
p_flood = ll[gmm_flood_id].subtract(ll_max).exp().divide(denom).rename("p_flood")

CUTOFFS = [0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.98]
pr_rows = []
for c in CUTOFFS:
    pr_rows.append({"cutoff": c, **score_flood(apply_post_masks(p_flood.gte(c)), urban=False)})
    print(f"  cut-off {c:g}: done")
pr_curve = pd.DataFrame(pr_rows)
pr_curve.to_csv(OUT_DIR / "pr_curve.csv", index=False)
pr_curve[["cutoff", "Flood mapped (km²)", "Precision", "Recall", "F1", "IoU"]].round(3)

### 8e. Figures

In [ ]:
%matplotlib inline
from matplotlib.lines import Line2D

fig, (ax_k, ax_pr) = plt.subplots(1, 2, figsize=(11, 4.3), layout="constrained")

# (a) Number of clusters
ks = cluster_choice["k"]
ax_k.plot(ks, cluster_choice["GMM BIC"] / 1e3, "o-", color=FLOOD_COLOR, ms=6, lw=1.6)
ax_k.set_xlabel("Number of clusters, k")
ax_k.set_ylabel("GMM BIC (×10³, lower is better)")
ax_k2 = ax_k.twinx()
ax_k2.plot(ks, cluster_choice["k-means inertia"] / 1e3, "s--", color="0.35", ms=5, lw=1.4)
ax_k2.set_ylabel("k-means within-cluster sum of squares (×10³)")
ax_k.axvline(N_CLUSTERS, color="black", ls=":", lw=1)
ax_k.text(N_CLUSTERS + 0.12, 0.95, f"k = {N_CLUSTERS} used", transform=ax_k.get_xaxis_transform(),
          va="top", fontsize=9)
ax_k.set_xticks(list(ks))
ax_k.grid(alpha=0.3)
ax_k.legend(handles=[
    Line2D([], [], color=FLOOD_COLOR, marker="o", lw=1.6, label="GMM BIC"),
    Line2D([], [], color="0.35", marker="s", ls="--", lw=1.4, label="k-means inertia"),
], loc="center right", fontsize=8, frameon=False)
ax_k.set_title("(a) Choosing the number of clusters", loc="left")

# (b) Precision-recall
ax_pr.plot(pr_curve["Recall"], pr_curve["Precision"], "o-", color=FLOOD_COLOR, ms=4, lw=1.6,
           label="GMM, varying P(flood) cut-off")
for c in (0.05, 0.2, 0.5, 0.9):
    r = pr_curve.loc[pr_curve["cutoff"] == c].iloc[0]
    ax_pr.annotate(f"{c:g}", (r["Recall"], r["Precision"]), xytext=(6, 5), textcoords="offset points",
                   fontsize=8, color="0.3")
points = [
    ("S1 GMM + masks (final)", "*", "#d62728", 14, "GMM hard label (final map)"),
    ("S1 k-means + masks", "D", "0.15", 6, "k-means + masks"),
    ("S1 Otsu on WI + masks", "^", "#1f77b4", 8, "Otsu threshold + masks"),
    ("S1 earlier rule (Otsu + cap + ΔWI) + masks", "s", "0.55", 6, "Earlier rule + masks"),
]
for row, marker, colour, size, label in points:
    r = comparison.loc[row]
    ax_pr.plot(r["Recall"], r["Precision"], marker, color=colour, ms=size, ls="none", label=label, zorder=5)
prec_all = np.r_[pr_curve["Precision"].values, [comparison.loc[p[0], "Precision"] for p in points]]
ax_pr.set_xlim(0, 1)
ax_pr.set_ylim(np.floor((np.nanmin(prec_all) - 0.02) * 20) / 20, 1.005)
ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.grid(alpha=0.3)
ax_pr.text(0.98, 0.98, "Labels: P(flood) cut-off", transform=ax_pr.transAxes, ha="right", va="top",
           fontsize=8, color="0.4")
ax_pr.legend(loc="lower left", fontsize=8, frameon=False)
ax_pr.set_title("(b) Precision–recall against the Dragonfly reference", loc="left")

fig.savefig(OUT_DIR / "method_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Ellipse

wi_db = X[:, 0] * wi_std + wi_mean
dw_db = X[:, 1] * dw_std + dw_mean
to_db_scale = np.diag([wi_std, dw_std])
OTHER_GREYS = ["0.25", "0.55", "0.78"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharex=True, sharey=True, layout="constrained")
panels = [
    (axes[0], gmm.predict(X), gmm_flood_id, "(a) Gaussian mixture model (used for map)"),
    (axes[1], kmeans.labels_, kmeans_flood_id, "(b) k-means (comparison)"),
]
for ax, labels, flood_id, title in panels:
    others = [k for k in range(N_CLUSTERS) if k != flood_id]
    colour = {flood_id: FLOOD_COLOR, **dict(zip(others, OTHER_GREYS))}
    for k in others + [flood_id]:
        sel = labels == k
        ax.scatter(wi_db[sel], dw_db[sel], s=3, color=colour[k], alpha=0.7, lw=0, rasterized=True)
    ax.axhline(0, color="black", ls="--", lw=0.8)
    ax.set_xlabel("Event water index, WI = VV + VH (dB)")
    ax.set_title(title, loc="left")
    ax.grid(alpha=0.3)

# GMM 2σ ellipses (Mahalanobis distance 2), converted from standardised units to dB.
for k in range(N_CLUSTERS):
    centre = gmm.means_[k] * [wi_std, dw_std] + [wi_mean, dw_mean]
    vals, vecs = np.linalg.eigh(to_db_scale @ gmm.covariances_[k] @ to_db_scale)
    angle = np.degrees(np.arctan2(vecs[1, -1], vecs[0, -1]))
    is_flood = k == gmm_flood_id
    axes[0].add_patch(Ellipse(centre, 4 * np.sqrt(vals[-1]), 4 * np.sqrt(vals[0]), angle=angle, fill=False,
                              color=FLOOD_COLOR if is_flood else "black", lw=1.8 if is_flood else 1.0, zorder=4))

centres = kmeans.cluster_centers_ * [wi_std, dw_std] + [wi_mean, dw_mean]
axes[1].scatter(centres[:, 0], centres[:, 1], marker="x", s=70, color="black", lw=1.8, zorder=5)
axes[0].set_ylabel("Change, ΔWI = WI$_{baseline}$ − WI$_{event}$ (dB)")

fig.legend(handles=[
    Line2D([], [], marker="o", ls="none", color=FLOOD_COLOR, label="Flood cluster"),
    Line2D([], [], marker="o", ls="none", color="0.55", label="Other clusters"),
    Line2D([], [], color="black", ls="--", lw=0.8, label="No change (ΔWI = 0)"),
    Line2D([], [], color="black", lw=1.0, label="GMM 2σ ellipse"),
    Line2D([], [], marker="x", ls="none", color="black", label="k-means centroid"),
], loc="outside lower center", ncol=5, frameon=False, fontsize=9)

fig.savefig(OUT_DIR / "feature_space.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Sample: {len(X)} pixels (permanent water excluded). Saved figures to {OUT_DIR}")

## Notes for the report
- **Urban gap.** Urban recall is near zero: flooded streets often look bright in SAR (double bounce), so a GMM component that needs **low WI and high ΔWI** still misses them. Affected-building counts downstream are therefore likely underestimates.
- **Single-scene snapshot.** Silt covers a larger area than either 14 Feb flood map, which suggests one SAR acquisition underestimates the maximum flood extent.
- **Silt model.** The Dragonfly silt layer could serve as training labels (split by area, not at random). Use S1 scenes after the water has receded, otherwise silt and water overlap.
- **Independence.** The HBRC flood map is the most independent reference; a stratified point check against it would complement section 4.